In [ ]:
!pip install mne
import mne
from mne.decoding import CSP
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

In [ ]:
with open("chb01-summary.txt", "r") as f:
    summary_text = f.read()

def get_seizure_times(summary_text, target_file):
    lines = summary_text.splitlines()
    seizure_times = []
    in_target = False

    for i, line in enumerate(lines):
        line = line.strip()

        if line.startswith("File Name:"):
            current_file = line.split(":")[1].strip()
            in_target = (current_file == target_file)

        if in_target:
            if "Number of Seizures in File: 0" in line:
                return []
            if "Seizure Start Time:" in line:
                start = int(line.split(":")[1].split()[0])
                end = int(lines[i+1].split(":")[1].split()[0])
                seizure_times.append((start, end))

    return seizure_times

In [ ]:
def process_file(file_path, seizure_times):
    raw = mne.io.read_raw_edf(file_path, preload=True)

    epochs = mne.make_fixed_length_epochs(raw, duration=2)
    data = epochs.get_data()

    labels = []

    for t in epochs.events[:, 0] / raw.info['sfreq']:
        label = 0
        for start, end in seizure_times:
            epoch_start = t
            epoch_end = t + 2

            if (epoch_start <= end) and (epoch_end >= start):
                label = 1
        labels.append(label)

    return data, np.array(labels)

In [ ]:
def build_dataset(file_list):
    X_all = []
    y_all = []

    for file in file_list:
        seizure_times = get_seizure_times(summary_text, file)
        X, y = process_file(file, seizure_times)

        X_all.append(X)
        y_all.append(y)

    return np.concatenate(X_all), np.concatenate(y_all)

In [ ]:
train_files = [
    "chb01_01.edf", "chb01_02.edf", "chb01_03.edf",
    "chb01_05.edf", "chb01_06.edf", "chb01_07.edf",
    "chb01_08.edf", "chb01_09.edf"
]

test_files = [
    "chb01_04.edf", "chb01_10.edf", "chb01_11.edf", "chb01_12.edf"
]

In [ ]:
X_train_raw, y_train = build_dataset(train_files)
X_test_raw, y_test = build_dataset(test_files)

print("Train labels:", np.unique(y_train, return_counts=True))
print("Test labels:", np.unique(y_test, return_counts=True))

In [ ]:
csp = CSP(n_components=4, reg='ledoit_wolf')

X_train_csp = csp.fit_transform(X_train_raw, y_train)
X_test_csp = csp.transform(X_test_raw)

In [ ]:
def create_sequences(X, y, seq_len=10):
    X_seq = []
    y_seq = []

    for i in range(len(X) - seq_len):
        seq = X[i:i+seq_len]
        label_window = y[i:i+seq_len]


        if np.sum(label_window) >= 2:
            label = 1
        else:
            label = 0

        X_seq.append(seq)
        y_seq.append(label)

    return np.array(X_seq), np.array(y_seq)

In [ ]:
X_train_seq, y_train_seq = create_sequences(X_train_csp, y_train)
X_test_seq, y_test_seq = create_sequences(X_test_csp, y_test)

In [ ]:
n_samples, seq_len, n_features = X_train_seq.shape

X_train_flat = X_train_seq.reshape(n_samples, seq_len * n_features)
X_test_flat = X_test_seq.reshape(
    X_test_seq.shape[0],
    seq_len * n_features
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

In [ ]:
model = LogisticRegression(class_weight='balanced')

model.fit(X_train_scaled, y_train_seq)

preds = model.predict(X_test_scaled)

In [ ]:
print(confusion_matrix(y_test_seq, preds))
print(classification_report(y_test_seq, preds))